In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as _sum, avg, round

# Membuat SparkSession
spark = SparkSession.builder \
    .appName("Tugas4_PySpark_HDFS") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [8]:
# Path file di HDFS sesuai petunjuk soal
hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"

# Membaca file CSV langsung dari HDFS
df = spark.read.csv(hdfs_path, header=True, inferSchema=True)

# 1. Menampilkan struktur/skema data
df.printSchema()

# 2. Menampilkan total jumlah baris
print(f"Jumlah total baris: {df.count()}")

# 3. Menampilkan 10 baris pertama
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah total baris: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|  

Penjelasan Bagian A:
1. arameter header=True digunakan agar baris pertama pada CSV dibaca sebagai nama kolom.
2. Parameter inferSchema=True membuat PySpark otomatis mendeteksi tipe data setiap kolom (seperti integer, double, atau string).
3. Fungsi count() menghitung total entri data, dan show(10) menampilkan 10 sampel baris teratas.

In [10]:
import builtins

# Hitung jumlah nilai null pada kolom rating
null_count = df.filter(col("rating").isNull()).count()
print(f"Jumlah data kosong di kolom 'rating': {null_count}")

# Menghitung rata-rata nilai rating yang ada
avg_rating = df.select(avg("rating")).first()[0]

# Mengisi nilai null dengan rata-rata rating (pakai builtins.round)
df_clean = df.na.fill({"rating": builtins.round(avg_rating, 2)})

# Verifikasi bahwa sudah tidak ada data kosong
print(f"Sisa data kosong di 'rating': {df_clean.filter(col('rating').isNull()).count()}")

Jumlah data kosong di kolom 'rating': 204
Sisa data kosong di 'rating': 0


Alasan memilih df.na.fill() dibanding df.na.drop():
Dalam analisis transaksi e-commerce, menghapus baris (drop) yang memiliki nilai rating kosong akan membuang informasi transaksi penting lainnya seperti unit_terjual dan harga_satuan. Hal ini dapat menyebabkan perhitungan total pendapatan (revenue) menjadi tidak akurat. Oleh karena itu, metode imputasi dengan df.na.fill() menggunakan nilai rata-rata (mean) lebih tepat agar volume transaksi tetap utuh tanpa merusak integritas analisis pendapatan.

In [11]:
# Menambahkan kolom total_pendapatan dan tier_transaksi
df_transformed = df_clean.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
                         .withColumn("tier_transaksi", when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))

# Menampilkan hasil transformasi
df_transformed.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

Penjelasan Bagian C:
1. withColumn() digunakan untuk membuat kolom baru.
2. Kolom total_pendapatan dihasilkan dari operasi aritmetika perkalian unit_terjual dengan harga_satuan.
3. Kolom tier_transaksi mengimplementasikan logika kondisional menggunakan when(kondisi, nilai_jika_benar).otherwise(nilai_jika_salah).

In [12]:
# 1. Kategori dengan total_pendapatan tertinggi
print("--- Kategori Pendapatan Tertinggi ---")
df_transformed.groupBy("kategori") \
    .agg(_sum("total_pendapatan").alias("total_pendapatan_kategori")) \
    .orderBy(col("total_pendapatan_kategori").desc()) \
    .show(1)

# 2. Kota dengan jumlah transaksi tier "Besar" terbanyak
print("--- Kota Transaksi 'Besar' Terbanyak ---")
df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("*").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc()) \
    .show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
print("--- Rata-rata Rating per Metode Pembayaran ---")
df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc()) \
    .show()

--- Kategori Pendapatan Tertinggi ---
+------------+-------------------------+
|    kategori|total_pendapatan_kategori|
+------------+-------------------------+
|Rumah Tangga|                138665000|
+------------+-------------------------+
only showing top 1 row

--- Kota Transaksi 'Besar' Terbanyak ---
+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                    92|
+----+----------------------+
only showing top 1 row

--- Rata-rata Rating per Metode Pembayaran ---
+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.168127490039839|
|    Transfer Bank|4.160079051383398|
|         E-Wallet|4.138599999999997|
|     Kartu Kredit|4.118902439024387|
+-----------------+-----------------+



Penjelasan Bagian D:
1. Mengelompokkan data berdasarkan kategori, menjumlahkan pendapatan dengan _sum(), diurutkan secara menurun (desc()), lalu mengambil 1 data teratas (show(1)).
2. Memfilter data yang hanya bertipe "Besar", mengelompokkan per kota, lalu menghitung frekuensi transaksinya dengan count().
3. Mengelompokkan data berdasarkan metode_pembayaran dan menghitung nilai rata-rata rating menggunakan avg().

In [13]:
# Path lokasi penyimpanan di HDFS
output_hdfs_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_transaksi_september_2026"

# Menyimpan DataFrame ke HDFS format CSV
df_transformed.write.mode("overwrite").csv(output_hdfs_path, header=True)

# Verifikasi: Membaca kembali data yang baru disimpan
df_verifikasi = spark.read.csv(output_hdfs_path, header=True)
df_verifikasi.show(5)

+--------+--------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|             tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+--------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02T00:00:...|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04T00:00:...|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26T00:00:...|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09T00:00:...|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|    

Mengapa Spark menyimpan hasil sebagai beberapa berkas partisi (part-00000...)?
Apache Spark dirancang dengan arsitektur komputasi terdistribusi (distributed computing). Saat proses penyimpanan data dilakukan, tiap-tiap worker node atau executor menulis potongan data (partition) miliknya secara independen dan bersamaan (parallel I/O) ke dalam HDFS. Pendekatan ini menghindari penumpukan beban pemrosesan (bottleneck) pada satu node tunggal sehingga pemrosesan Big Data dapat berjalan sangat cepat.